In [1]:
# For text preprocessing
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel
import pandas as pd

In [2]:
df = pd.read_csv('news_dataset.csv')
# Remove rows with missing values in 'text'
df = df.dropna(subset=['text'])
documents = df['text'].tolist()

In [16]:
stop_words = set(stopwords.words('english'))
# Add custom stopwords
extra_stopwords = {'dont', 'would', 'get', 'know', 'think', 'like', 'one', 'even', 'also', 'well'}
stop_words.update(extra_stopwords)

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Lowercase
    text = text.lower()
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    # Filter out non-alphanumeric tokens
    tokens = [token for token in tokens if token.isalnum()]
    # Remove very short tokens
    tokens = [word for word in tokens if len(word) > 2]
    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens

# Apply preprocessing
preprocessed_docs = [preprocess_text(doc) for doc in documents]

In [17]:
# Create a Gensim Dictionary object from the preprocessed documents
dictionary = corpora.Dictionary(preprocessed_docs)

# Filter out tokens that appear in less than 15 documents or more than 50% of the documents
dictionary.filter_extremes(no_below=15, no_above=0.5)

# Convert each preprocessed document into a bag-of-words representation using the dictionary
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_docs] 

In [22]:
# Run LDA
lda_model = LdaModel(
    corpus, 
    num_topics=4, 
    id2word=dictionary, 
    passes=15) 
# Train an LDA model on the corpus with 4 topics using Gensim's LdaModel class

In [23]:
coherence_model = CoherenceModel(
    model=lda_model,
    texts=preprocessed_docs,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()
print(f"\nCoherence Score (c_v): {coherence_score:.4f}")


Coherence Score (c_v): 0.5135


In [24]:
# Assign dominant topic to each document
topic_labels = []
for doc in preprocessed_docs:
    bow = dictionary.doc2bow(doc)
    topics = lda_model.get_document_topics(bow)
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    topic_labels.append(dominant_topic)

# Create a list of cleaned text strings (space‑joined tokens)
cleaned_texts = [" ".join(doc) for doc in preprocessed_docs]

# Then use that in the DataFrame
result_df = pd.DataFrame({
    "Cleaned_Text_Snippet": [text[:100] + "..." for text in cleaned_texts],
    "Assigned_Topic": topic_labels
})

print("\nSample of assigned topics (first 10 rows):")
print(result_df.head(10))


Sample of assigned topics (first 10 rows):
                                Cleaned_Text_Snippet  Assigned_Topic
0  wondering anyone could enlighten car saw day d...               0
1  recently posted article asking kind rate singl...               0
2  depends priority lot people put higher priorit...               0
3  excellent automatic found subaru legacy switch...               0
4  ford automobile need information whether ford ...               0
5  watch attributionsi didnt say isnt appropriate...               1
6  avoid problem entirely installing oil drain va...               0
7  acura integra speed mile positively worst car ...               0
8  assuming non turbo gruffness characteristic la...               0
9  addition restricted mileage many classic insur...               0


In [25]:
# Print top terms for each topic
print("\nTop terms per topic:")
for topic_id in range(lda_model.num_topics):
    top_terms = lda_model.show_topic(topic_id, topn=20)
    terms = [term[0] for term in top_terms]
    print(f"Topic {topic_id}: {', '.join(terms)}")



Top terms per topic:
Topic 0: window, use, problem, drive, work, thanks, bit, anyone, need, time, system, good, card, could, used, please, ive, want, disk, much
Topic 1: people, say, right, god, time, government, thing, make, many, could, way, see, state, said, believe, want, law, mean, something, armenian
Topic 2: year, game, president, team, new, last, first, time, player, going, play, good, said, two, back, russian, day, april, job, season
Topic 3: key, file, system, use, program, information, encryption, chip, available, data, number, clipper, message, privacy, user, may, security, public, anonymous, technology


Topic 0: Computer System
Topic 1: Social Issues
Topic 2: Sports
Topic 3: Cybersecurity

The coherence score of 0.5135 confirms that all four topics are well‑separated, where a term within each topic frequently reoccurs in the same document. 
However, Topic 2 has some mix of topics like sports and politics, but it might be due to the noise.
